## Feature selection

In this notebook, features used to train the models were selected and constructed from the cleaned dataset. We used two main approaches:

- Term Frequency-Inverse Document Frequency (TF-IDF)
- Word2Vec Embeddings

These two approaches are compared on a Logistic Regressor and the best approach is then chosen for the next step.

### Importing dependencies and loading cleaned data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from gensim.models import Word2Vec

import nltk
nltk.download('stopwords', quiet=True)

train_df = pd.read_csv("Dataset/train.csv")
test_df  = pd.read_csv("Dataset/test.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nColumns:", train_df.columns.tolist())
print("\nLabel distribution (train):")
print(train_df["label"].value_counts().rename({0: "Fake", 1: "True"}))

Train shape: (33043, 6)
Test shape : (11015, 6)

Columns: ['title', 'text', 'cleaned_title', 'cleaned_text', 'content', 'label']

Label distribution (train):
label
Fake    17136
True    15907
Name: count, dtype: int64


### Approach 1: TF-IDF

This technique is used to capture words that appear in one document but rarely across all documents. This strategy generates representation of pure statistical word(phrase) importance.

How it works: a fixed vocabulary of 50000 frequent and unique words(phrases) is built and scored using the `TFxIDF` formula.

We used L2 normalization because longer articles are not needed to dominatejust by having more words.

In [ ]:
from sklearn.preprocessing import normalize

# --- Body-only TF-IDF ---
tfidf_body = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), norm="l2")
X_train_body = tfidf_body.fit_transform(train_df["cleaned_text"])
X_test_body  = tfidf_body.transform(test_df["cleaned_text"])

# --- Title+Body TF-IDF ---
tfidf_combined = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), norm="l2")
X_train_combined = tfidf_combined.fit_transform(train_df["content"])
X_test_combined  = tfidf_combined.transform(test_df["content"])

y_train = train_df["label"]
y_test  = test_df["label"]

print("Body-only matrix    :", X_train_body.shape)
print("Title+Body matrix   :", X_train_combined.shape)
print("L2 norm sample (body row 0):", X_train_body[0].toarray().sum()**0.5)

Body-only matrix    : (33043, 50000)
Title+Body matrix   : (33043, 50000)
L2 norm sample (body row 0): 3.6778509253418203


### Using cross validation to tune hyperparameters

Next, the hyperparameters C and ngram_range are selected together with comparing the usage of news body only to title + body.

F1 is used as a performance measure because of the information it gives on precision and recall. Mean and standard deviation of F1 score is used to rank the hyperparameters' performance for both `news article` and `article with title` features.

In [4]:
from sklearn.model_selection import cross_val_score

configs = [
    {"name": "body  | (1,1) | C=0.1",  "X": train_df["cleaned_text"], "ngram": (1,1), "C": 0.1},
    {"name": "body  | (1,2) | C=0.1",  "X": train_df["cleaned_text"], "ngram": (1,2), "C": 0.1},
    {"name": "body  | (1,2) | C=1.0",  "X": train_df["cleaned_text"], "ngram": (1,2), "C": 1.0},
    {"name": "body  | (1,2) | C=10.0", "X": train_df["cleaned_text"], "ngram": (1,2), "C": 10.0},
    {"name": "comb  | (1,1) | C=0.1",  "X": train_df["content"],      "ngram": (1,1), "C": 0.1},
    {"name": "comb  | (1,2) | C=0.1",  "X": train_df["content"],      "ngram": (1,2), "C": 0.1},
    {"name": "comb  | (1,2) | C=1.0",  "X": train_df["content"],      "ngram": (1,2), "C": 1.0},
    {"name": "comb  | (1,2) | C=10.0", "X": train_df["content"],      "ngram": (1,2), "C": 10.0},
]

results = []
for cfg in configs:
    vec = TfidfVectorizer(max_features=50000, ngram_range=cfg["ngram"], norm="l2")
    X   = vec.fit_transform(cfg["X"])
    clf = LogisticRegression(C=cfg["C"], max_iter=1000, solver="lbfgs", n_jobs=-1)
    scores = cross_val_score(clf, X, y_train, cv=5, scoring="f1", n_jobs=-1)
    results.append({"config": cfg["name"], "mean_f1": scores.mean(), "std_f1": scores.std()})
    print(f"{cfg['name']}  →  F1: {scores.mean():.4f} ± {scores.std():.4f}")

results_df = pd.DataFrame(results).sort_values("mean_f1", ascending=False)
print("\n--- Ranked ---")
print(results_df.to_string(index=False))


body  | (1,1) | C=0.1  →  F1: 0.9671 ± 0.0034
body  | (1,2) | C=0.1  →  F1: 0.9700 ± 0.0033
body  | (1,2) | C=1.0  →  F1: 0.9862 ± 0.0028
body  | (1,2) | C=10.0  →  F1: 0.9923 ± 0.0025
comb  | (1,1) | C=0.1  →  F1: 0.9671 ± 0.0030
comb  | (1,2) | C=0.1  →  F1: 0.9703 ± 0.0028
comb  | (1,2) | C=1.0  →  F1: 0.9870 ± 0.0028
comb  | (1,2) | C=10.0  →  F1: 0.9930 ± 0.0024

--- Ranked ---
                config  mean_f1   std_f1
comb  | (1,2) | C=10.0 0.992964 0.002445
body  | (1,2) | C=10.0 0.992308 0.002472
 comb  | (1,2) | C=1.0 0.986987 0.002799
 body  | (1,2) | C=1.0 0.986171 0.002810
 comb  | (1,2) | C=0.1 0.970323 0.002755
 body  | (1,2) | C=0.1 0.969985 0.003299
 body  | (1,1) | C=0.1 0.967109 0.003370
 comb  | (1,1) | C=0.1 0.967064 0.003019


Approach 2: Word2Vec Embeddings: training a small Neural Network to learn same meaning words appear in similar contexts.

In [5]:
from sklearn.preprocessing import StandardScaler

# Tokenize for Word2Vec (expects list of token lists)
train_tokens_body = [text.split() for text in train_df["cleaned_text"]]
train_tokens_comb = [text.split() for text in train_df["content"]]
test_tokens_body  = [text.split() for text in test_df["cleaned_text"]]
test_tokens_comb  = [text.split() for text in test_df["content"]]

# Train Word2Vec models
print("Training Word2Vec on body...")
w2v_body = Word2Vec(sentences=train_tokens_body, vector_size=100, window=5,
                    min_count=2, workers=4, epochs=5, seed=42)

print("Training Word2Vec on combined...")
w2v_comb = Word2Vec(sentences=train_tokens_comb, vector_size=100, window=5,
                    min_count=2, workers=4, epochs=5, seed=42)

def average_vectors(token_lists, model):
    """Average word vectors for each document; zero vector if no known words."""
    vecs = []
    for tokens in token_lists:
        known = [model.wv[t] for t in tokens if t in model.wv]
        vecs.append(np.mean(known, axis=0) if known else np.zeros(model.vector_size))
    return np.array(vecs)

# Build document matrices
print("Building document vectors...")
X_train_w2v_body = average_vectors(train_tokens_body, w2v_body)
X_test_w2v_body  = average_vectors(test_tokens_body,  w2v_body)
X_train_w2v_comb = average_vectors(train_tokens_comb, w2v_comb)
X_test_w2v_comb  = average_vectors(test_tokens_comb,  w2v_comb)

# StandardScaler
scaler_body = StandardScaler()
X_train_w2v_body = scaler_body.fit_transform(X_train_w2v_body)
X_test_w2v_body  = scaler_body.transform(X_test_w2v_body)

scaler_comb = StandardScaler()
X_train_w2v_comb = scaler_comb.fit_transform(X_train_w2v_comb)
X_test_w2v_comb  = scaler_comb.transform(X_test_w2v_comb)

print(f"W2V body matrix  : {X_train_w2v_body.shape}")
print(f"W2V combined matrix: {X_train_w2v_comb.shape}")
print("Done.")


Training Word2Vec on body...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Training Word2Vec on combined...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Building document vectors...
W2V body matrix  : (33043, 100)
W2V combined matrix: (33043, 100)
Done.


Comparison of the configurations using the test data. Keep in mind we are not selecting a hyperparameter here. We are just using our feature translations to create a Logistic regression model and evaluate its performance to see how it does with unseen data.

In [6]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(X_train, X_test, y_train, y_test, C=1.0):
    clf = LogisticRegression(C=C, max_iter=1000, solver="lbfgs", n_jobs=-1)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, preds),
        "f1":       f1_score(y_test, preds)
    }

# Use best C from Cell 3 results — adjust if your CV showed a different winner
best_C = 10.0

comparison = [
    ("TF-IDF | body-only  ", X_train_body,     X_test_body,     best_C),
    ("TF-IDF | title+body ", X_train_combined,  X_test_combined, best_C),
    ("Word2Vec | body-only ", X_train_w2v_body,  X_test_w2v_body, best_C),
    ("Word2Vec | title+body", X_train_w2v_comb,  X_test_w2v_comb, best_C),
]

rows = []
for name, Xtr, Xte, C in comparison:
    metrics = evaluate(Xtr, Xte, y_train, y_test, C)
    rows.append({"config": name, "accuracy": metrics["accuracy"], "f1": metrics["f1"]})
    print(f"{name}  →  Acc: {metrics['accuracy']:.4f}  F1: {metrics['f1']:.4f}")

comp_df = pd.DataFrame(rows).sort_values("f1", ascending=False)
print("\n--- Ranked by F1 ---")
print(comp_df.to_string(index=False))

best_config = comp_df.iloc[0]["config"].strip()
print(f"\nBest config: {best_config}")


TF-IDF | body-only    →  Acc: 0.9953  F1: 0.9951
TF-IDF | title+body   →  Acc: 0.9939  F1: 0.9937
Word2Vec | body-only   →  Acc: 0.9695  F1: 0.9685
Word2Vec | title+body  →  Acc: 0.9735  F1: 0.9726

--- Ranked by F1 ---
               config  accuracy       f1
 TF-IDF | body-only    0.995279 0.995096
 TF-IDF | title+body   0.993917 0.993688
Word2Vec | title+body  0.973491 0.972603
Word2Vec | body-only   0.969496 0.968528

Best config: TF-IDF | body-only


Saving best config

In [13]:
import joblib
import json
import os

os.makedirs("models", exist_ok=True)

# --- Determine best config from Cell 5 ---
# Update these manually if your results differed
BEST_FEATURE  = "tfidf"        # "tfidf" or "w2v"
BEST_INPUT    = "body"     # "body" or "combined"
BEST_NGRAM    = (1, 2)
BEST_C        = 10.0

# Save the winning vectorizer/scaler
if BEST_FEATURE == "tfidf":
    vec = tfidf_combined if BEST_INPUT == "combined" else tfidf_body
    joblib.dump(vec, "models/best_vectorizer.pkl")
    print("Saved TF-IDF vectorizer → models/best_vectorizer.pkl")
else:
    model = w2v_comb if BEST_INPUT == "combined" else w2v_body
    scaler = scaler_comb if BEST_INPUT == "combined" else scaler_body
    model.save("models/best_w2v.model")
    joblib.dump(scaler, "models/best_scaler.pkl")
    print("Saved Word2Vec model → models/best_w2v.model")
    print("Saved scaler         → models/best_scaler.pkl")

# Save config metadata for Notebook 3 to read
config = {
    "feature_type":  BEST_FEATURE,
    "input_field":   "content" if BEST_INPUT == "combined" else "cleaned_text",
    "ngram_range":   list(BEST_NGRAM),
    "best_C":        BEST_C,
    "max_features":  50000,
    "vector_size":   100
}
with open("models/best_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("\nSaved config → models/best_config.json")
print(json.dumps(config, indent=2))


Saved TF-IDF vectorizer → models/best_vectorizer.pkl

Saved config → models/best_config.json
{
  "feature_type": "tfidf",
  "input_field": "cleaned_text",
  "ngram_range": [
    1,
    2
  ],
  "best_C": 10.0,
  "max_features": 50000,
  "vector_size": 100
}


Summary

In [12]:
print("=" * 55)
print("     NOTEBOOK 2 — FEATURE SELECTION SUMMARY")
print("=" * 55)

with open("models/best_config.json") as f:
    cfg = json.load(f)

print(f"""
FEATURE SETS COMPARED
---------------------
  1. TF-IDF  | body-only   (cleaned_text)
  2. TF-IDF  | title+body  (content)
  3. Word2Vec | body-only   (cleaned_text)
  4. Word2Vec | title+body  (content)

PREPROCESSING
-------------
  TF-IDF   : max_features=50,000, L2 normalization
  Word2Vec  : vector_size=100, window=5, avg pooling, StandardScaler

HYPERPARAMETER TUNING
---------------------
  Method    : 5-fold cross-validation on training set
  Params    : ngram_range ∈ {{(1,1),(1,2)}}, C ∈ {{0.1, 1.0, 10.0}}
  Metric    : F1 score

BEST CONFIGURATION
------------------
  Feature type  : {cfg['feature_type'].upper()}
  Input field   : {cfg['input_field']}
  Ngram range   : {cfg['ngram_range']}
  Best C        : {cfg['best_C']}

SAVED ARTIFACTS
---------------
  models/best_vectorizer.pkl  (or best_w2v.model + best_scaler.pkl)
  models/best_config.json
""")
print("=" * 55)


     NOTEBOOK 2 — FEATURE SELECTION SUMMARY

FEATURE SETS COMPARED
---------------------
  1. TF-IDF  | body-only   (cleaned_text)
  2. TF-IDF  | title+body  (content)
  3. Word2Vec | body-only   (cleaned_text)
  4. Word2Vec | title+body  (content)

PREPROCESSING
-------------
  TF-IDF   : max_features=50,000, L2 normalization
  Word2Vec  : vector_size=100, window=5, avg pooling, StandardScaler

HYPERPARAMETER TUNING
---------------------
  Method    : 5-fold cross-validation on training set
  Params    : ngram_range ∈ {(1,1),(1,2)}, C ∈ {0.1, 1.0, 10.0}
  Metric    : F1 score

BEST CONFIGURATION
------------------
  Feature type  : TFIDF
  Input field   : content
  Ngram range   : [1, 2]
  Best C        : 10.0

SAVED ARTIFACTS
---------------
  models/best_vectorizer.pkl  (or best_w2v.model + best_scaler.pkl)
  models/best_config.json

